# Notebook 02 — Patient-Level Splits + Verification

**Run after Notebook 01.**

Creates stratified 70/10/20 patient-level splits.
Verifies zero leakage between train/val/test.
Shows class balance per split.

In [1]:
# ─────────────────────────────────────────────
# CELL 1 — Load paths and records
# ─────────────────────────────────────────────
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd

# ── Project paths ────────────────────────────
PROJECT_ROOT = Path('.').resolve().parent

paths = json.loads(
    (PROJECT_ROOT / 'paths.json').read_text()
)

PREPROCESSED = Path(paths['preprocessed'])

# ── Splits directory ─────────────────────────
SPLITS_DIR = PROJECT_ROOT / 'splits'
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

# ── Load preprocessing records ───────────────
df = pd.read_csv(
    PREPROCESSED / 'records.csv'
)

print(f'Total slices  : {len(df)}')
print(f'Total patients: {df["patient_id"].nunique()}')

print('\nClass distribution:')
print(df.groupby('label')['patient_id'].nunique())

print('\nCenters:')
print(df.groupby('center')['patient_id'].nunique())

Total slices  : 3075
Total patients: 221

Class distribution:
label
0    142
1     83
Name: patient_id, dtype: int64

Centers:
center
1    119
2     47
3     24
4     31
Name: patient_id, dtype: int64


In [2]:
# ─── Patient-level stratified split ──────────────
SEED = 42
rng  = np.random.default_rng(SEED)

patient_df = (
    df.drop_duplicates('patient_id')
    [['patient_id', 'center', 'label']]
    .copy()
)
patient_df['strat'] = (
    patient_df['center'].astype(str) + '_' +
    patient_df['label'].astype(str)
)

train_pids, val_pids, test_pids = [], [], []

for key, grp in patient_df.groupby('strat'):
    pids = grp['patient_id'].tolist()
    rng.shuffle(pids)
    n = len(pids)
    n_test = max(1, round(n * 0.20))
    n_val  = max(1, round(n * 0.10))
    n_train = n - n_test - n_val
    if n_train < 1:
        train_pids.extend(pids); continue
    train_pids.extend(pids[:n_train])
    val_pids.extend(pids[n_train:n_train+n_val])
    test_pids.extend(pids[n_train+n_val:])

train_df = df[df['patient_id'].isin(set(train_pids))].copy()
val_df   = df[df['patient_id'].isin(set(val_pids))].copy()
test_df  = df[df['patient_id'].isin(set(test_pids))].copy()

# ─── Verify zero leakage ─────────────────────────
tp = set(train_df.patient_id); vp = set(val_df.patient_id); ep = set(test_df.patient_id)
assert len(tp & vp) == 0, 'LEAKAGE train/val'
assert len(tp & ep) == 0, 'LEAKAGE train/test'
assert len(vp & ep) == 0, 'LEAKAGE val/test'
print('No leakage ✓')

# ─── Save splits ─────────────────────────────────
train_df.to_csv(SPLITS_DIR / 'train.csv', index=False)
val_df.to_csv  (SPLITS_DIR / 'val.csv',   index=False)
test_df.to_csv (SPLITS_DIR / 'test.csv',  index=False)

print(f'\nSplit summary (patient-level stratified 70/10/20):')
for name, df_ in [('train', train_df), ('val', val_df), ('test', test_df)]:
    pts  = df_.drop_duplicates('patient_id')
    mibc = (pts['label']==1).sum()
    print(f'  {name:5s}: {len(pts):3d} patients | '
          f'{len(df_):4d} slices | '
          f'MIBC={mibc} ({100*mibc/len(pts):.0f}%)')
print(f'\nSplits saved to {SPLITS_DIR}/')

No leakage ✓

Split summary (patient-level stratified 70/10/20):
  train: 157 patients | 2285 slices | MIBC=57 (36%)
  val  :  22 patients |  267 slices | MIBC=8 (36%)
  test :  42 patients |  523 slices | MIBC=15 (36%)

Splits saved to C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\splits/
